In [ ]:
# Biblioteca pandas para análise de dados.
import pandas as pd

# Lendo os CNPJs como texto (str) desde o início.
# Antes, ao ler o CSV sem especificar o tipo, o pandas podia interpretar
# essas colunas como número e perder o zero à esquerda em uma leitura futura
dtype_cnpj = {
    "cnpj_instituicao": str,
    "cnpj_fornecedor": str,
    "cnpj_fabricante": str,
}

# Carregando o arquivo CSV.
df = pd.read_csv(
    "BPS_20_26_LuizFernandoDeJesusSilvaHomem.csv",
    sep=";",
    encoding="utf-8",
    low_memory=False,
    dtype=dtype_cnpj,
)

# Mudando o jeito como o DataFrame é exibido.
pd.set_option("display.max_columns", None)

# Conferindo se os CNPJs vieram como texto
print(df[["cnpj_instituicao", "cnpj_fornecedor", "cnpj_fabricante"]].dtypes)

# Visualizando as primeiras linhas.
df.head()

In [ ]:
# Identificando o tamanho do DataFrame.
print(f"Tamanho do DataFrame: {df.shape[0]} linhas e {df.shape[1]} colunas")
# Identificando o tipo de dados de cada coluna.
print(f"\nTipo de dados de cada coluna:\n{df.dtypes}")

In [ ]:
# Comfirmando visualmente que as colunas de preços e capacidade são numéricas.
df[["preco_unitario", "preco_total", "capacidade"]].describe().round(2)

In [ ]:
# Comfirmando visualmente que as colunas de datas são do tipo datetime.
df[["compra", "insercao", "validade_compra"]].head(10)

In [ ]:
# Convertendo as colunas de datas para o tipo datetime.
df["compra"] = pd.to_datetime(df["compra"], format="%d/%m/%Y", errors="coerce")
df["insercao"] = pd.to_datetime(df["insercao"], format="%d/%m/%Y", errors="coerce")

# Conferindo o resultado.
print(f"Novo tipo de dados de cada coluna:\n{df.dtypes}")
df[["compra", "insercao"]].head(10)

In [ ]:
# Conferindo se a conversão gerou valores nulos nas colunas de datas.
print(f'Nulos em "compra": {df["compra"].isnull().sum()}')
print(f'Nulos em "insercao": {df["insercao"].isnull().sum()}')

In [ ]:
# Identificando quais linhas ficaram nulas em "insercao" após a conversão.
linhas_nulas = df[df["insercao"].isnull()]

# Recarregando a coluna original (antes da conversão) para comparar.
df_original = pd.read_csv(
    "BPS_20_26_LuizFernandoDeJesusSilvaHomem.csv",
    sep=";",
    encoding="utf-8",
    low_memory=False,
    usecols=["insercao"],
)

# pegando os valores originais correspondentes às linhas problemáticas.
valores_originais_problematicos = df_original.loc[linhas_nulas.index, "insercao"]

print(valores_originais_problematicos.value_counts(dropna=False).head(20))

In [ ]:
# Mapeando os valores nulos em todoas as colunas.
nulos = df.isnull().sum()
percentual_nulos = (nulos / len(df) * 100).round(2)

resumo_nulos = pd.DataFrame({"qtd_nulos": nulos, "percentual_nulos": percentual_nulos})

# Mostrando só as colunas que possuem valores nulos.
resumo_nulos[resumo_nulos["qtd_nulos"] > 0].sort_values(
    by="percentual_nulos", ascending=False
)

resumo_nulos

In [ ]:
# Investigando nulos nas colunas:
colunas_investigar = [
    "generico",
    "anvisa",
    "unidade_medida",
    "capacidade",
    "nu_ata",
    "nome_instituicao",
]

for col in colunas_investigar:
    print(f"\n--- {col} ---")
    print(df.groupby("ano_compra")[col].apply(lambda x: x.isnull().sum()))

In [ ]:
print(
    "generico e anvisa são nulos nas mesma linha?",
    (df["generico"].isnull() == df["anvisa"].isnull()).all(),
)

print(
    "unidade_medida e capacidade são nulos nas mesma linha?",
    (df["unidade_medida"].isnull() == df["capacidade"].isnull()).all(),
)

In [ ]:
# Olhando os valores não-nulos de nu_ata.
preenchidos = df[df["nu_ata"].notnull()]

print(f"Total de registros com nu_ata preenchido: {len(preenchidos)}")
print(f"\nDistribuição por ano:")
print(preenchidos["ano_compra"].value_counts().sort_index())

print(f"\nExemplos de valores:")
print(preenchidos["nu_ata"].head(10).tolist())

print(f"\nComparando o preco_total médio: preenchidos vs. vazios")
print("Preenchidos:", preenchidos["preco_total"].mean().round(2))
print("Vazios:", df[df["nu_ata"].isnull()]["preco_total"].mean().round(2))

In [ ]:
# Descartando a coluna nu_ata, pois ela não tem relevância para a análise e possui muitos valores nulos.
df = df.drop(columns=["nu_ata"])
print(df.shape)

In [ ]:
# Olhando as linhas com nome_instituicao nulo.
sem_instituicao = df[df["nome_instituicao"].isnull()]

print(f"Total de linhas sem nome_instituicao: {len(sem_instituicao)}")

# cnpj_instituicao está nessas linhas?
print(
    f"\ncnpj_instituicao nulo: {sem_instituicao['cnpj_instituicao'].isnull().sum()} de {len(sem_instituicao)}"
)

# Olhando alguns CNPJs das linhas problematicas.
print(f"\nCNPJs das instituições sem nome:")
print(sem_instituicao["cnpj_instituicao"].value_counts().head(10))

In [ ]:
# Criando um mapa de CNPJ -> nome_instituicao, usando só linhas onde o nome já existe.
mapa_cnpj_nome = (
    df[df["nome_instituicao"].notnull()]
    .drop_duplicates(subset="cnpj_instituicao")
    .set_index("cnpj_instituicao")["nome_instituicao"]
)

# Verificando quantos dos CNPJs "sem nome" da para recuperar.
cnpjs_sem_nome = sem_instituicao["cnpj_instituicao"].unique()
recuperaveis = [cnpj for cnpj in cnpjs_sem_nome if cnpj in mapa_cnpj_nome.index]

print(f"CNPJs distintos sem nome: {len(cnpjs_sem_nome)}")
print(
    f"Desses, quantos aparecem em outra linha com o nome preenchido: {len(recuperaveis)}"
)

In [ ]:
# Preenchendo os nome ausentes usando o mapa CNPJ -> nome
df["nome_instituicao"] = df["nome_instituicao"].fillna(
    df["cnpj_instituicao"].map(mapa_cnpj_nome)
)

# Conferindo se ainda restou algum nulo
print(f"Nulos restantes em nome_instituição: {df['nome_instituicao'].isnull().sum()}")

In [ ]:
# Verificando se são nulos nas mesma linhas
print(
    "unidade_fornecimento e unidade_consumo são nulos nas mesmas linhas:",
    (
        df["unidade_fornecimento"].isnull()
        == df["unidade_fornecimento_capacidade"].isnull()
    ).all(),
)

# Quem são essas linhas.
sem_unidade = df[df["unidade_fornecimento"].isnull()]
print(f"\nTotal de linhas: {len(sem_unidade)}")
print(f"\nDistribuição por ano:")
print(sem_unidade["ano_compra"].value_counts().sort_index())

print(f"\nExibindo algumas linhas:")
print(
    sem_unidade[
        ["ano_compra", "descricao_catmat", "unidade_medida", "capacidade"]
    ].head(10)
)

In [ ]:
# Verificando duplicatas exatas (todas as colunas iguais).
duplicatas_exatas = df.duplicated(keep=False)

print(f"Total de linhas duplicadaas (exatas): {duplicatas_exatas.sum()}")
print(f"Percentual da Base: {(duplicatas_exatas.sum() / len(df) * 100):.2f}%")

In [ ]:
# Quem são essas linhas duplicadas.
colunas_chave = [
    "ano_compra",
    "cnpj_instituicao",
    "descricao_catmat",
    "compra",
    "fornecedor",
    "preco_total",
    "qtd_itens_comprados",
]

print(df[duplicatas_exatas][colunas_chave].sort_values(by=colunas_chave))

In [ ]:
# Removendo Duplicatas exatas, mantendo a primeira ocorrencia de cada.
df = df.drop_duplicates(keep="first")

print(f"Novo tamanho da base: {df.shape}")

In [ ]:
# Duplicatas parciais.
colunas_chaves_parcial = [
    "ano_compra",
    "cnpj_instituicao",
    "codigo_br",
    "compra",
    "cnpj_fornecedor",
    "qtd_itens_comprados",
]

duplicatas_parciais = df.duplicated(subset=colunas_chaves_parcial, keep=False)

print(f"Total de linhas com possivel duplicata parcial: {duplicatas_parciais.sum()}")
print(f"Percentual da base: {(duplicatas_parciais.sum() / len(df) * 100):.2f}%")

In [ ]:
# Quem são essas colunas parciais.
colunas_ver = [
    "ano_compra",
    "cnpj_instituicao",
    "codigo_br",
    "descricao_catmat",
    "compra",
    "cnpj_fornecedor",
    "qtd_itens_comprados",
    "preco_unitario",
    "preco_total",
    "unidade_fornecimento",
]

amostra = df[duplicatas_parciais][colunas_ver].sort_values(
    by=["ano_compra", "cnpj_instituicao", "codigo_br", "compra"]
)

print(amostra.head(30))

In [ ]:
# Dentro das duplicatas parciais, isolando só os pares onde o preço também bate
duplicatas_parciais_com_preco_igual = df[duplicatas_parciais].duplicated(
    subset=colunas_chaves_parcial + ["preco_unitario", "preco_total"], keep=False
)

grupo_suspeito = df[duplicatas_parciais][duplicatas_parciais_com_preco_igual]
print(f"Linhas com mesma chave e mesmo preço: {len(grupo_suspeito)}")

# Exibindo as linhas completas lado a lado, para achar a diferença.
print(grupo_suspeito.head(2).T)

In [ ]:
# Conferindo o tipo de dado e o tamanho dos CNPJs
print(df["cnpj_instituicao"].dtype)
print(df["cnpj_fornecedor"].dtype)
print(df["cnpj_fabricante"].dtype)

# CNPJ tem 14 digitos. Quais são os tamanhos (em caracteres) de cada um deles.
print("\nTamanho dos cnpj_insituicao (contagem por nº de digitos:):")
print(df["cnpj_instituicao"].astype(str).str.len().value_counts())

In [ ]:
# Confirmando Visualmente esses erros.
print("Exemplos com 13 caracteres (faltando 1 zero):")
print(
    df[df["cnpj_instituicao"].astype(str).str.len() == 13]["cnpj_instituicao"]
    .head(5)
    .tolist()
)

print("\nExemplos com 12 caracteres (faltando 2 zeros):")
print(
    df[df["cnpj_instituicao"].astype(str).str.len() == 12]["cnpj_instituicao"]
    .head(5)
    .tolist()
)

print("\nExemplos com 18 caracteres (com pontuação):")
print(
    df[df["cnpj_instituicao"].astype(str).str.len() == 18]["cnpj_instituicao"]
    .head(5)
    .tolist()
)

In [ ]:
def normalizar_cnpj(coluna):
    # Remove qualquer caractere que não seja digito (ponto, barra, traço)
    cnpj_limpo = coluna.astype(str).str.replace(r"\D", "", regex=True)
    # Completa com zeros à esquerda até ter 14 digitos
    cnpj_padronizado = cnpj_limpo.str.zfill(14)
    return cnpj_padronizado


df["cnpj_instituicao"] = normalizar_cnpj(df["cnpj_instituicao"])
df["cnpj_fornecedor"] = normalizar_cnpj(df["cnpj_fornecedor"])
df["cnpj_fabricante"] = normalizar_cnpj(df["cnpj_fabricante"])

# Conferindo se todos ficaram com 14 caracteres agora.
print(df["cnpj_instituicao"].str.len().value_counts())
print(df["cnpj_fornecedor"].str.len().value_counts())
print(df["cnpj_fabricante"].str.len().value_counts())

In [ ]:
colunas_texto = df.select_dtypes(include="object").columns.tolist()

for col in colunas_texto:
    # Procurando sinais de encoding quebrados.
    suspeitos = df[col].astype(str).str.contains(r"Ã|Â|�", regex=True, na=False)
    total_suspeitos = suspeitos.sum()
    if total_suspeitos > 0:
        print(f"{col}: {total_suspeitos} valores suspeitos")

In [ ]:
# Olhando exemplos reais dos valores "suspeitos" em descricao_catmat
suspeitos_descricao = df[
    df["descricao_catmat"].astype(str).str.contains(r"Ã|Â|�", regex=True, na=False)
]

print(suspeitos_descricao["descricao_catmat"].head(15).tolist())

In [ ]:
# Para cada CNPJ, contar quantos nomes distintos aparecem associados a ele.
nomes_por_cnpj = df.groupby("cnpj_instituicao")["nome_instituicao"].nunique()

# Filtrando só os CNPJs que têm mais de 1 nome diferente.
cnpjs_com_nome_diferentes = nomes_por_cnpj[nomes_por_cnpj > 1]

print(
    f"CNPJs de instituição com mais de uma nome distinto: {len(cnpjs_com_nome_diferentes)}"
)
print(f"Total de CNPJs distintos na base: {df['cnpj_instituicao'].nunique()}")

In [ ]:
# Quem são esses CNPJs distintos.
cnpjs_investigar = cnpjs_com_nome_diferentes.index.tolist()

for cnpj in cnpjs_investigar:
    nomes = df[df["cnpj_instituicao"] == cnpj]["nome_instituicao"].unique()
    print(f"\nCNPJ: {cnpj}")
    print(f"Nomes encontrados: {nomes}")

In [ ]:
cnpjs_padronizar = ["11295659000139", "12157728000100", "18478187000107"]

for cnpj in cnpjs_padronizar:
    nome_mais_frequente = df[df["cnpj_instituicao"] == cnpj]["nome_instituicao"].mode()[
        0
    ]
    df.loc[df["cnpj_instituicao"] == cnpj, "nome_instituicao"] = nome_mais_frequente
    print(f"{cnpj} -> padronizado para: {nome_mais_frequente}")

# Conferindo que ficou 1 nome só nesses 3 CNPJs
for cnpj in cnpjs_padronizar:
    print(df[df["cnpj_instituicao"] == cnpj]["nome_instituicao"].nunique())

In [ ]:
# Removendo espaços duplos (ou mais) em nome_instituicao
df["nome_instituicao"] = (
    df["nome_instituicao"].str.replace(r"\s+", " ", regex=True).str.strip()
)

# Conferindo o resultado
print(df[df["cnpj_instituicao"] == "18478187000107"]["nome_instituicao"].unique())

In [ ]:
# Mesma checagem para fornecedor
nomes_por_cnpj_fornecedor = df.groupby("cnpj_fornecedor")["fornecedor"].nunique()
cnpjs_fornecedor_variando = nomes_por_cnpj_fornecedor[nomes_por_cnpj_fornecedor > 1]

print(
    f"CNPJs de fornecedor com mais de um nome distinto: {len(cnpjs_fornecedor_variando)}"
)
print(f"Total de CNPJs de fornecedor distintos: {df['cnpj_fornecedor'].nunique()}")

# Mesma checagem para fabricante
nomes_por_cnpj_fabricante = df.groupby("cnpj_fabricante")["fabricante"].nunique()
cnpjs_fabricante_variando = nomes_por_cnpj_fabricante[nomes_por_cnpj_fabricante > 1]

print(
    f"\nCNPJs de fabricante com mais de um nome distinto: {len(cnpjs_fabricante_variando)}"
)
print(f"Total de CNPJs de fabricante distintos: {df['cnpj_fabricante'].nunique()}")

In [ ]:
print("=== FORNECEDOR ===")
for cnpj in cnpjs_fornecedor_variando.index.tolist():
    nomes = df[df["cnpj_fornecedor"] == cnpj]["fornecedor"].unique()
    print(f"\nCNPJ: {cnpj}")
    print(f"Nomes: {nomes}")

print("\n\n=== FABRICANTE ===")
for cnpj in cnpjs_fabricante_variando.index.tolist():
    nomes = df[df["cnpj_fabricante"] == cnpj]["fabricante"].unique()
    print(f"\nCNPJ: {cnpj}")
    print(f"Nomes: {nomes}")

In [ ]:
# Aplicando a mesma limpeza de espaços duplos em fornecedor e fabricante
df["fornecedor"] = df["fornecedor"].str.replace(r"\s+", " ", regex=True).str.strip()
df["fabricante"] = df["fabricante"].str.replace(r"\s+", " ", regex=True).str.strip()

# Conferindo se os nomes convergiram para um único valor por CNPJ
print(
    "Fornecedores com mais de 1 nome após limpeza:",
    (df.groupby("cnpj_fornecedor")["fornecedor"].nunique() > 1).sum(),
)

print(
    "Fabricantes com mais de 1 nome após limpeza:",
    (df.groupby("cnpj_fabricante")["fabricante"].nunique() > 1).sum(),
)

In [ ]:
# Olhando as colunas de códigos que foram lidas como float64
colunas_codigo = ["co_pdm", "co_grupo", "co_classe", "anvisa", "co_seq_bps"]

for col in colunas_codigo:
    print(f"{col}: {df[col].dropna().head(3).tolist()}")

In [ ]:
# Convertendo para Int64 pois se tiver campos vazios irá reconhecer como nulo.
colunas_codigo = ["co_pdm", "co_grupo", "co_classe", "anvisa", "co_seq_bps"]

for col in colunas_codigo:
    df[col] = df[col].astype("Int64")

# Conferindo o resultado
for col in colunas_codigo:
    print(f"{col}: {df[col].dropna().head(3).tolist()}, dtype: {df[col].dtype}")

In [ ]:
# Olhando a coluna de validade da compra e tentando achar outros valores além do 12.
print(df["validade_compra"].value_counts(dropna=False))

In [ ]:
# Investigando os 4 registros com validade_compra = 90
print(
    df[df["validade_compra"] == 90][
        [
            "ano_compra",
            "nome_instituicao",
            "descricao_catmat",
            "compra",
            "modalidade_compra",
        ]
    ]
)

In [ ]:
# Checando quantos registros nulos temos em cada coluna (essas colunas pertencem apenas a tabela do ano de 2020)
colunas_classe = ["co_pdm", "co_grupo", "no_grupo", "co_classe", "no_classe", "no_pdm"]

# Filtrando só as linhas de 2020
df_2020 = df[df["ano_compra"] == 2020]

print(f"Total de linhas em 2020: {len(df_2020)}\n")

for col in colunas_classe:
    nulos_2020 = df_2020[col].isnull().sum()
    percentual = (nulos_2020 / len(df_2020) * 100).round(2)
    print(f"{col}: {nulos_2020} nulos em 2020 ({percentual}%)")

In [ ]:
# Finalizando a limpeza e fechando com o salvamento do DataFrame.
saida_final = "BPS_20_26_LuizFernandoDeJesusSilvaHomem.csv"

df.to_csv(saida_final, index=False, sep=";", encoding="utf-8")

print(f"Base salva como: {saida_final}")
print(f"Linhas: {len(df)} | Colunas: {len(df.columns)}")

In [ ]:
# 1º KPI
# Criando um KPI valor total registrado em python para validar antes do DataStudio.
valor_total_registrado = df["preco_total"].sum()

print(f"valor total registrado: R$ {valor_total_registrado:,.2f}")

In [ ]:
# 2º KPI
# Quantidade total de itens comprados.
quantidade_total_itens = df["qtd_itens_comprados"].sum()

print(f"Quantidade total de itens comprados: {quantidade_total_itens:,}")

In [ ]:
# Checando porque deu um número tão grande assim.
print(df["qtd_itens_comprados"].describe())

print("\nTop 10 maiores quantidades:")
print(
    df.nlargest(10, "qtd_itens_comprados")[
        [
            "ano_compra",
            "descricao_catmat",
            "qtd_itens_comprados",
            "unidade_fornecimento",
            "nome_instituicao",
        ]
    ]
)

In [ ]:
# Confirmando os 10 maiores quantidades
top10 = df.nlargest(10, "qtd_itens_comprados")[
    [
        "ano_compra",
        "descricao_catmat",
        "qtd_itens_comprados",
        "preco_unitario",
        "preco_total",
    ]
].copy()

top10["preco_total_calculado"] = top10["qtd_itens_comprados"] * top10["preco_unitario"]
top10["bate_com_preco_total"] = top10["preco_total_calculado"].round(2) == top10[
    "preco_total"
].round(2)

print(top10)

In [ ]:
# Checando o top 10 completo
top10_completo = df.nlargest(10, "qtd_itens_comprados")[
    [
        "ano_compra",
        "descricao_catmat",
        "qtd_itens_comprados",
        "modalidade_compra",
        "nome_instituicao",
    ]
]

print(top10_completo)

In [ ]:
# Quanto os 10 maiores valores representam do total da soma?
top10_soma = df.nlargest(10, "qtd_itens_comprados")["qtd_itens_comprados"].sum()
total_geral = df["qtd_itens_comprados"].sum()

print(f"Soma dos 10 maiores valores: {top10_soma:,}")
print(f"Soma total da base: {total_geral:,}")
print(
    f"Percentual que os 10 maiores representam do total: {(top10_soma/total_geral*100):.2f}%"
)

In [ ]:
# 3º KPI número de registros de compra.
numero_registros = len(df)

print(f"Número de registros de compra: {numero_registros:,}")

In [ ]:
# 4º KPI Instituições Compradoras.
instituicoes_distintas = df["cnpj_instituicao"].nunique()

print(f"Instituições compradoras (distintas): {instituicoes_distintas}")

In [ ]:
# 5º KPI Fornecedores.
fornecedores_distintos = df["cnpj_fornecedor"].nunique()

print(f"Fornecedores (distintos): {fornecedores_distintos}")

In [ ]:
# 6º KPI Preço unitário médio Ponderado.

preco_medio_ponderado = df["preco_total"].sum() / df["qtd_itens_comprados"].sum()

print(f"Preço unitário médio ponderado: R$ {preco_medio_ponderado:.4f}")

# Comparando com a média simples (sem ponderação), só para referência
media_simples = df["preco_unitario"].mean()
print(f"Média simples (sem ponderação), só para comparação: R$ {media_simples:.4f}")